# Lecture 1: How Agents Really Work (hands-on)
**Strands Agents with Ollama · Lecture 1 of 5** · companion to `Lecture01_How_Agents_Really_Work.pdf`

You will:
1. Run a plain agent on a local Ollama model
2. Turn Python functions into tools with `@tool`
3. Build a tool-using agent with a system prompt
4. Read `agent.messages` and metrics to see what the loop did
5. Experiment with small-model knobs (thinking mode, model size)

## 0. Setup: Ollama + Strands
Same steps as `StrandsAgent_SetUp.ipynb`. Run these once per Colab session (about 2 to 3 minutes).

> **Tip:** *Runtime → Change runtime type → T4 GPU* makes Ollama much faster. CPU works too, just slower.

In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq zstd
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background and wait until it answers
import subprocess, time, urllib.request
subprocess.Popen("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start - check /tmp/ollama.log")

In [ ]:
!ollama pull qwen3:0.6b
# Optional, more reliable for tool use (1.4 GB):
# !ollama pull qwen3:1.7b

In [ ]:
!pip install -q "strands-agents[ollama,web-fetch]"

### Shared model helper
Every lecture uses the same `make_model()` helper. Change `MODEL_ID` here to switch models for the whole notebook.

In [ ]:
from strands.models.ollama import OllamaModel

MODEL_ID = "qwen3:0.6b"          # try "qwen3:1.7b" for better tool use

def make_model(model_id=MODEL_ID, **kwargs):
    return OllamaModel(
        host="http://127.0.0.1:11434",
        model_id=model_id,
        options={"num_ctx": 8192},   # explicit context window (see Lecture 4)
        **kwargs,
    )

import importlib.metadata as md_
print("strands-agents", md_.version("strands-agents"), "| model:", MODEL_ID)

## 1. A simple agent
`Agent(model=...)` is already a working agent with a loop, just without tools. The default callback streams text as it's generated.

In [ ]:
from strands import Agent

agent = Agent(model=make_model())
response = agent("What are the key differences between REST and GraphQL APIs?")

In [ ]:
# The return value is an AgentResult, not just a string
print(type(response).__name__)
print("stop_reason:", response.stop_reason)
print("usage:", response.metrics.accumulated_usage)

## 2. Turning a function into a tool
The **docstring** becomes the tool description and the **type hints + Args** become the input schema. Let's print exactly what the model sees.

In [ ]:
from strands import Agent, tool

@tool
def query_product_database(query: str) -> str:
    """Query the internal product database for inventory and pricing information.

    Args:
        query: Search query for products (e.g., "wireless headphones", "USB-C hub")
    """
    products = {
        "wireless headphones": "SKU-WH100: Wireless Headphones Pro - $79.99, 142 in stock, 4.5★ rating, launched 2025-03",
        "usb-c hub": "SKU-UC200: USB-C Hub 7-in-1 - $45.00, 89 in stock, 4.2★ rating, launched 2024-11",
        "mechanical keyboard": "SKU-MK300: Mechanical Keyboard RGB - $149.99, 23 in stock, 4.8★ rating, launched 2025-01",
        "noise cancelling": "SKU-NC400: Noise Cancelling Earbuds - $129.99, 67 in stock, 4.6★ rating, launched 2025-05",
    }
    key = query.lower()
    matches = [info for product_key, info in products.items() if product_key in key]
    if matches:
        return "\n".join(matches)
    return f"No products found matching '{query}'. Available: wireless headphones, usb-c hub, mechanical keyboard, noise cancelling"

query_product_database.tool_spec

In [ ]:
# Tools are still normal Python functions - test them directly first
print(query_product_database(query="wireless headphones"))

## 3. Watching the agent loop
Ask a question that needs the tool, then inspect `agent.messages`. You should see: `user text → assistant toolUse → user toolResult → assistant text`.

In [ ]:
agent = Agent(
    model=make_model(),
    tools=[query_product_database],
    system_prompt="You are a product analyst. Always use query_product_database to answer product questions.",
)
result = agent("What is the price of our wireless headphones?")

In [ ]:
for i, m in enumerate(agent.messages):
    for block in m["content"]:
        kind = list(block.keys())[0]
        detail = ""
        if kind == "toolUse":
            detail = f"{block['toolUse']['name']}({block['toolUse']['input']})"
        elif kind == "toolResult":
            detail = str(block["toolResult"]["content"])[:80]
        elif kind == "text":
            detail = block["text"][:80].replace("\n", " ")
        print(f"{i:>2} {m['role']:<10} {kind:<11} {detail}")

print("\nstop_reason:", result.stop_reason)

In [ ]:
summary = result.metrics.get_summary()
print("cycles:", summary["total_cycles"])
print("tool usage:", summary["tool_usage"])
print("tokens:", summary["accumulated_usage"])

## 4. Agent with vended + custom tools
This is the example from the official lesson, running on Ollama. `web_fetch` and `file_editor` ship with Strands.

> **Small-model tip:** in the setup run, `qwen3:0.6b` asked for a URL instead of searching. Giving it a concrete URL makes it far more reliable.

In [ ]:
from strands.vended_tools import file_editor
from strands.vended_tools.web_fetch import web_fetch

SYSTEM_PROMPT = """You are a product research analyst. You help the team understand
market positioning by comparing competitor pricing with our internal catalog.

When given a research task:
1. Use web_fetch to gather public market data from the web
2. Use query_product_database to check our internal pricing and inventory
3. Write a brief competitive analysis and save it to report.md using file_editor"""

agent = Agent(
    model=make_model(),
    tools=[web_fetch, file_editor, query_product_database],
    system_prompt=SYSTEM_PROMPT,
)

result = agent("Fetch https://en.wikipedia.org/wiki/Headphones to learn about headphone market trends, "
               "then compare against our wireless headphones offering. "
               "Write a short competitive positioning summary and save it to report.md")

In [ ]:
import os
print(open("report.md").read() if os.path.exists("report.md") else "report.md was not written - inspect agent.messages to see why")

## 5. Small-model knobs
Measured in testing on CPU: `think=False` was about 9× faster, but the 0.6b model then **skipped the tool**. Try it yourself.

In [ ]:
import time

def trial(model_id, think):
    a = Agent(model=make_model(model_id, additional_args={"think": think}),
              tools=[query_product_database], callback_handler=None,
              system_prompt="Always use query_product_database to answer product questions.")
    t = time.time()
    r = a("What is the price of our mechanical keyboard?")
    used = any("toolUse" in b for m in a.messages for b in m["content"])
    print(f"{model_id:<11} think={str(think):<5} {time.time()-t:5.1f}s  tool_used={used}  -> {str(r).strip()[:70]}")

trial("qwen3:0.6b", True)
trial("qwen3:0.6b", False)
# trial("qwen3:1.7b", True)    # after: !ollama pull qwen3:1.7b

## 6. Your turn
**Exercise 1:** Add a tool `check_competitor_price(product: str) -> str` that returns made-up competitor prices. Ask a question that needs **both** tools and confirm both appear in `agent.messages`.

In [ ]:
@tool
def check_competitor_price(product: str) -> str:
    """TODO: write a clear docstring - the model reads this!

    Args:
        product: TODO
    """
    # TODO: return made-up competitor prices
    ...

# TODO: build an agent with both tools and ask a comparison question

**Exercise 2:** Make the docstring of `check_competitor_price` deliberately vague (e.g. `"""Does stuff."""`). Does the model still call it? Restore it afterwards.

**Exercise 3:** Re-run section 5 with `qwen3:1.7b` and fill in the table:

| Model | think | seconds | tool used? |
|---|---|---|---|
| qwen3:0.6b | True | | |
| qwen3:0.6b | False | | |
| qwen3:1.7b | True | | |

In [ ]:
# Your experiments here